# Video Anomaly Detection on Google Colab

MSc Data Science final project (24EMDT7218). This notebook runs the whole pipeline on a free Colab GPU:
UCSD Ped2 download → integrity check → EDA → PCA / CAE / masked ViT-AE training → evaluation.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Put `video-anomaly-detection.zip` in Google Drive at `MyDrive/MSc_VAD/` (or set `SOURCE = "github"` below once the repo exists).
3. Run the cells top to bottom (`Runtime → Run all` works too).

Results and trained models are copied back to `MyDrive/MSc_VAD/results_<date>/` at the end, so they survive the Colab session.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
import torch; print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive and get the code

In [ ]:
# "drive"  -> unzip MyDrive/MSc_VAD/video-anomaly-detection.zip  (works today)
# "github" -> clone github.com/nevin7771/video-anomaly-detection (after you create and push the repo;
#             for a private repo add a Colab secret named GITHUB_TOKEN via the key icon on the left)
SOURCE = "drive"
DRIVE_DIR = "/content/drive/MyDrive/MSc_VAD"

from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess
os.makedirs(DRIVE_DIR, exist_ok=True)
PROJECT = "/content/video-anomaly-detection"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)

if SOURCE == "drive":
    zip_path = f"{DRIVE_DIR}/video-anomaly-detection.zip"
    assert os.path.exists(zip_path), f"Upload video-anomaly-detection.zip to {DRIVE_DIR} first"
    subprocess.run(["unzip", "-q", zip_path, "-d", "/content"], check=True)
else:
    url = "https://github.com/nevin7771/video-anomaly-detection.git"
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = f"https://{token}@github.com/nevin7771/video-anomaly-detection.git"
    except Exception:
        pass  # public repo, no token needed
    subprocess.run(["git", "clone", "-q", url, PROJECT], check=True)

%cd {PROJECT}
!ls

## 3. Install dependencies
PyTorch is already installed on Colab with CUDA, so only the extra packages are added.

In [ ]:
!pip install -q opencv-python-headless scipy scikit-learn matplotlib pyyaml

## 4. Download and verify UCSD Ped2 (~460 MB)

In [ ]:
!bash scripts/download_ped2.sh
!python scripts/prepare_data.py

## 5. Exploratory data analysis

In [ ]:
!python scripts/eda.py > /dev/null
from IPython.display import Image, display
for f in ["eda_sample_frames", "eda_motion_energy", "eda_anomaly_area", "eda_anomaly_prevalence"]:
    display(Image(f"results/figures/{f}.png", width=800))

## 6. Train and score the models

`QUICK = True` runs 2 epochs and 1 seed (a few minutes) to check everything works.
`QUICK = False` is the full Review 2 experiment (PCA, CAE with and without motion, masked ViT-AE, 3 seeds), about 15–25 min on a T4.

In [ ]:
QUICK = False
EP = "--epochs 2" if QUICK else ""
SEEDS = [0] if QUICK else [0, 1, 2]

!python scripts/train.py --model pca
!python scripts/train.py --model cae --no-motion --seed 0 {EP}
for s in SEEDS:
    !python scripts/train.py --model cae --seed {s} {EP}
    !python scripts/train.py --model vit --seed {s} {EP}

## 7. Evaluate

In [ ]:
!python scripts/evaluate.py
from IPython.display import Markdown
display(Markdown(open("results/metrics/results_table.md").read()))
for f in ["auc_by_model", "roc_curves", "score_timelines", "localisation_heatmaps"]:
    display(Image(f"results/figures/{f}.png", width=850))

## 8. Save results and trained models to Google Drive

In [ ]:
import datetime
out = f"{DRIVE_DIR}/results_{datetime.datetime.now():%Y%m%d_%H%M}"
shutil.copytree("results", f"{out}/results", ignore=shutil.ignore_patterns("scores"))
shutil.copytree("checkpoints", f"{out}/checkpoints")
print("Saved to", out)

## 9. (Optional) Rebuild the Review 2 slides with these results

Writes `Video_Anomaly_Detection_Second_Review.pptx` to the same Drive folder.

In [ ]:
!VAD_THEME=dark python scripts/eda.py > /dev/null
!VAD_THEME=dark python scripts/evaluate.py > /dev/null
!cd slides && npm install --silent && node build_deck.js
shutil.copy("slides/Video_Anomaly_Detection_Second_Review.pptx", out)
print("Slides saved to", out)

## Experimenting

* Change settings in `configs/ped2.yaml` (image size, clip length, epochs), then re-run sections 6–7.
* Train one model: `!python scripts/train.py --model vit --seed 0 --epochs 40`
* Colab disconnects after inactivity; section 8 is what keeps your results.